# Proyecto 1: EDA + Baseline

**Ciencia de Datos, Sección A** · Segundo semestre 2026

Entrega: **viernes 11 de septiembre** · 10 puntos

---

**Autor(es):** Philip Falla

**Dataset elegido:** Predict Students' Dropout and Academic Success (UCI Machine Learning Repository, dataset ID 697)

### Rúbrica resumida

| Criterio | Pts | Qué buscamos |
|---|---|---|
| Pregunta y datos | 2 | Pregunta clara, provenance del dataset |
| EDA | 3 | Hallazgos, no galería de gráficas |
| Features | 2 | Cada decisión justificada, sin leakage |
| Baseline | 2 | Validación correcta, métrica adecuada |
| Límites | 1 | Qué no puede concluirse con estos datos |

## 1. Pregunta y contexto

Las instituciones de educación superior suelen identificar tarde a los estudiantes en riesgo de abandonar: para cuando las notas de fin de semestre confirman el problema, ya pasó una cohorte entera de matrícula sin intervención. Si una universidad pudiera estimar ese riesgo usando solo la información que ya tiene el día en que un estudiante se inscribe —antes de que exista un solo registro de desempeño académico—, podría dirigir tutorías, becas o seguimiento a quienes más lo necesitan desde el primer semestre, en lugar de esperar a que el abandono ya esté en curso.

Este proyecto pregunta qué tanto explica esa información personal y socioeconómica —sin incluir desempeño curricular ni datos de admisión— la probabilidad de que un estudiante abandone la carrera. La respuesta le importa a una oficina de bienestar o retención estudiantil: si un modelo entrenado solo con este tipo de variables apenas supera predecir siempre "no abandona", la conclusión práctica es que el perfil personal no basta y hay que esperar señales académicas tempranas (primer parcial, asistencia) para intervenir con criterio. Si en cambio distingue con margen claro sobre ese baseline trivial, se justifica invertir en programas de apoyo dirigidos desde el momento de la admisión.

**Pregunta:** ¿Qué tanto explica la información personal y socioeconómica de un estudiante (género, edad, estado civil, nacionalidad, condición de desplazado, necesidades educativas especiales, educación y ocupación de los padres, si es deudor, si tiene las cuotas al día, si es becado, si es estudiante internacional) su probabilidad de abandonar la carrera, frente a simplemente predecir siempre la clase mayoritaria?

**Tipo de problema:** Clasificación binaria.

**Variable objetivo:** `Target`, recodificada como `dropout` (1 = Dropout, 0 = Enrolled o Graduate). Se colapsan "Enrolled" y "Graduate" en una sola clase de "no abandono" porque la pregunta es específicamente sobre abandono, no sobre distinguir entre seguir inscrito y graduarse.

## 2. Los datos

- **Fuente:** UCI Machine Learning Repository, dataset "Predict Students' Dropout and Academic Success" (ID 697). Realinho, V., Vieira Martins, M., Machado, J., & Baptista, L. (2021). https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success — DOI: https://doi.org/10.24432/C5MC89
- **Fecha de descarga:** 14 de septiembre de 2026
- **Licencia:** CC BY 4.0 (uso y redistribución permitidos citando la fuente)
- **Unidad de observación:** cada fila es un estudiante matriculado en una carrera de grado de una institución de educación superior portuguesa (varios programas: agronomía, diseño, enfermería, gestión, periodismo, servicio social, tecnologías, entre otros), con cohortes de ingreso entre 2008/09 y 2018/19. El dataset combina datos conocidos al momento de la matrícula (demográficos, socioeconómicos, de admisión) con el desempeño académico al final del 1er y 2do semestre, y el estatus final del estudiante.

**Diccionario breve — variables que usaremos como predictores (información personal y socioeconómica, según la sección 1):**

| Variable | Tipo | Descripción |
|---|---|---|
| `Marital status` | categórica (código) | Estado civil al matricularse |
| `Nacionality` | categórica (código) | Nacionalidad |
| `Displaced` | binaria | Si el estudiante se considera desplazado de su residencia habitual |
| `Educational special needs` | binaria | Si declara necesidades educativas especiales |
| `Debtor` | binaria | Si tiene deudas pendientes con la institución |
| `Tuition fees up to date` | binaria | Si las cuotas de matrícula están al día |
| `Gender` | binaria | Género (codificación original: 0 = femenino, 1 = masculino) |
| `Scholarship holder` | binaria | Si recibe beca |
| `Age at enrollment` | entera | Edad del estudiante al matricularse |
| `International` | binaria | Si es estudiante internacional |
| `Mother's qualification` / `Father's qualification` | categórica (código) | Nivel educativo más alto de la madre / del padre |
| `Mother's occupation` / `Father's occupation` | categórica (código) | Categoría ocupacional de la madre / del padre |

**Variable objetivo derivada:**

| Variable | Tipo | Descripción |
|---|---|---|
| `Target` → `dropout` | binaria (derivada) | 1 si `Target == "Dropout"`, 0 si `Target` es `"Enrolled"` o `"Graduate"` |

Las demás columnas (admisión, desempeño curricular por semestre, indicadores macroeconómicos) se cargan junto con el resto del dataset para el primer barrido, pero quedan fuera de los predictores del modelo por la razón explicada en la sección 1.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42

# Cargar el dataset (descargado de la fuente oficial UCI, ver sección 2)
df = pd.read_csv("data/data.csv", sep=";")
df.columns = df.columns.str.strip()  # el CSV original trae un tab pegado al nombre de una columna

# Variable objetivo binaria definida en la sección 1: 1 = abandonó, 0 = sigue inscrito o se graduó
df["dropout"] = (df["Target"] == "Dropout").astype(int)

df.shape

## 3. Primer barrido

In [ ]:
# df.shape, df.info(), df.head()
print(df.shape)
df.info()
df.head()

In [ ]:
# df.describe() para numéricas, value_counts() para categóricas
df[["Age at enrollment"]].describe()

In [ ]:
# value_counts() de las variables categóricas/personales que usaremos como predictores
personales = ["Marital status", "Nacionality", "Displaced", "Educational special needs",
              "Debtor", "Tuition fees up to date", "Gender", "Scholarship holder", "International"]

for col in personales:
    print(df[col].value_counts().sort_index())
    print()

**Observaciones del primer barrido:**

1. El dataset viene completamente numérico (30 columnas `int64`, 7 `float64`) salvo `Target` (texto). Esto incluye a las variables categóricas que nos interesan (`Marital status`, `Nacionality`, calificación y ocupación de los padres): están codificadas como enteros, no como texto, así que en la sección 6 hay que tratarlas explícitamente como categóricas (`OneHotEncoder`) y no dejar que el pipeline las interprete como numéricas continuas con orden o magnitud.

2. `Nacionality` está extremadamente desbalanceada: 4,314 de 4,424 estudiantes (97.5%) tienen el código 1 (Portugal); el resto se reparte en ~20 categorías con conteos ínfimos (varias con 1-3 estudiantes). Tal cual, esta columna aporta casi cero varianza como predictor y con un one-hot directo generaría columnas casi vacías — candidata a colapsar en "nacional / extranjero" en features.

3. `Marital status` también está concentrada: 88.6% en la categoría 1 (soltero); las categorías 3 y 6 tienen menos de 10 observaciones cada una. Mismo problema de niveles raros que `Nacionality`, aunque menos extremo.

4. Las variables binarias personales (`Displaced`, `Educational special needs`, `Debtor`, `Tuition fees up to date`, `Gender`, `Scholarship holder`, `International`) están limpias, sin valores fuera de {0,1} y sin nulos — no requieren limpieza adicional.

5. `Age at enrollment` tiene media 23.3 y mediana 20 (rango 17-70, desviación estándar 7.6): la distribución tiene cola larga hacia edades mayores (estudiantes que regresan a estudiar), no es simétrica. Vale la pena revisar esa cola en la sección 4 de outliers antes de decidir si se recorta o se deja tal cual.

## 4. Calidad de datos

In [ ]:
# Faltantes por columna, en cantidad y en porcentaje
print("NaNs explícitos en todo el dataset:", df.isna().sum().sum())

# El dataset no trae NaN, pero el diccionario de datos oficial de UCI documenta
# códigos que representan "desconocido" / "en blanco" dentro de variables categóricas
faltantes_disfrazados = {
    "Mother's qualification": 34,  # 34 = "Unknown"
    "Father's qualification": 34,  # 34 = "Unknown"
    "Mother's occupation": 99,     # 99 = "(blank)"
    "Father's occupation": 99,     # 99 = "(blank)"
}

for col, code in faltantes_disfrazados.items():
    n = (df[col] == code).sum()
    print(f"{col}: {n} filas con código {code} ({n / len(df) * 100:.2f}%)")

In [ ]:
# Duplicados
print("Filas duplicadas:", df.duplicated().sum())

# Outliers en Age at enrollment (IQR)
q1, q3 = df["Age at enrollment"].quantile([0.25, 0.75])
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers_edad = (df["Age at enrollment"] < lo) | (df["Age at enrollment"] > hi)
print(f"Outliers de edad (IQR): {outliers_edad.sum()} filas ({outliers_edad.mean() * 100:.2f}%), límites [{lo:.0f}, {hi:.0f}]")

# ¿La "ausencia" de datos de los padres se relaciona con el abandono?
mask_desconocido = (
    (df["Mother's qualification"] == 34) | (df["Father's qualification"] == 34)
    | (df["Mother's occupation"] == 99) | (df["Father's occupation"] == 99)
)
print("\nTarget entre filas con dato de padres 'desconocido/en blanco':")
print(df.loc[mask_desconocido, "Target"].value_counts(normalize=True))
print("\nTarget en todo el dataset:")
print(df["Target"].value_counts(normalize=True))

**Mecanismo de los faltantes disfrazados:** no son MCAR. Entre las 148 filas (3.3% del dataset) donde al menos uno de los cuatro campos de los padres viene "Unknown"/"(blank)", el 73% son `Dropout`, frente a 32% en el dataset completo — la probabilidad de que falte el dato está asociada con la variable objetivo, no es aleatoria. No podemos confirmar si esa asociación se explica del todo por otras variables observadas (lo que sería MAR) o si depende de algo no capturado en el propio proceso de abandono (MNAR), así que tratamos el mecanismo como MAR con sospecha de MNAR y evitamos cualquier imputación que asuma aleatoriedad (media, moda, KNN-imputer).

**Decisiones tomadas y su justificación:**

| Problema | Filas o columnas afectadas | Qué hicimos | Por qué |
|---|---|---|---|
| Faltantes disfrazados (código "Unknown"/"(blank)") en calificación/ocupación de los padres | 148 filas (3.3%) con al menos uno de los 4 campos afectado | Se dejan como una categoría explícita más al codificar (one-hot), sin imputar ni eliminar filas | Como se argumenta arriba, la ausencia está asociada al abandono (MAR/MNAR); imputar con la media o la moda, o eliminar esas filas, borraría justo la señal que más nos interesa |
| Outliers de edad (>34 años, límite superior IQR) | 441 filas (10.0%) | Se conservan sin modificar | No son errores de captura: son estudiantes adultos que regresan a estudiar, un grupo real ya visible en la sección 3 (cola derecha de la distribución). Recortarlos sesgaría el modelo contra ese segmento |
| Duplicados | 0 filas | Ninguna acción | El dataset no tiene filas duplicadas |
| `Nacionality` con 97.5% en una sola categoría (sección 3) | 1 columna | Se colapsa a binaria `is_portuguese` en la sección 6 de features | Las ~20 categorías restantes tienen 1-14 observaciones cada una; un one-hot directo crearía columnas casi vacías sin aportar señal generalizable |

## 5. Análisis exploratorio

In [ ]:
# Distribución de la variable objetivo
print(df["Target"].value_counts(normalize=True).round(3))
print("\ndropout binario:")
print(df["dropout"].value_counts(normalize=True).round(3))

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(data=df, x="Target", order=["Graduate", "Dropout", "Enrolled"], ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# Relaciones entre predictores personales y objetivo
binarias = ["Displaced", "Debtor", "Tuition fees up to date", "Gender", "Scholarship holder", "International"]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flat, binarias):
    tasa = df.groupby(col)["dropout"].mean()
    sns.barplot(x=tasa.index.astype(str), y=tasa.values, ax=ax)
    ax.set_title(col)
    ax.set_ylabel("tasa de dropout")
    ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

# Edad al matricularse, por dropout
fig, ax = plt.subplots(figsize=(5, 4))
sns.boxplot(data=df, x="dropout", y="Age at enrollment", ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# Matriz de correlación (recordar: correlación no es causalidad)
df["is_portuguese"] = (df["Nacionality"] == 1).astype(int)

cols_corr = ["Age at enrollment", "Displaced", "Educational special needs", "Debtor",
             "Tuition fees up to date", "Gender", "Scholarship holder", "International",
             "Marital status", "is_portuguese", "dropout"]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df[cols_corr].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
plt.tight_layout()
plt.show()

**Hallazgos del EDA (numerados, con la gráfica que los respalda):**

1. **Distribución del target (gráfica de barras del target):** 49.9% Graduate, 32.1% Dropout, 18.0% Enrolled. Es un desbalance moderado, no extremo (no es el caso 95/5 que exigiría cambiar de métrica): el baseline trivial de "siempre no-dropout" acierta 67.9% de las veces, así que cualquier modelo tiene que superar ese número, no un 50%.

2. **`Tuition fees up to date` (panel de barras):** es el predictor personal con más separación de todo el set. 86.6% de dropout entre quienes NO tienen las cuotas al día, contra 24.7% entre quienes sí (r = -0.43 con `dropout` en el mapa de calor) — la variable individual más fuerte de la sección.

3. **`Debtor` y `Scholarship holder` (panel de barras):** son dos caras de la misma situación económica. Ser deudor casi duplica la tasa de dropout (62.0% vs 28.3%, r = 0.23); tener beca la reduce a menos de un tercio (12.2% vs 38.7%, r = -0.25).

4. **`Gender` y `Age at enrollment` (panel de barras y boxplot):** los hombres abandonan más que las mujeres en este dataset (45.1% vs 25.1%, r = 0.20). Quienes abandonan se matriculan en promedio 4 años más tarde (26.1 vs 21.9 años; mediana 23 vs 19) y con más dispersión — la caja de `dropout=1` está claramente desplazada hacia arriba en el boxplot.

5. **Redundancia entre `International` y `is_portuguese` (mapa de calor):** ambas correlacionan perfectamente en sentido inverso (r = -1.00). No es casualidad: en este dataset los 110 estudiantes internacionales son exactamente los 110 no-portugueses, así que son la misma partición con otro nombre. Hay que usar solo una de las dos en la sección 6 para no duplicar la misma señal.

6. **Ninguna variable individual basta por sí sola (mapa de calor):** la correlación más fuerte con `dropout` es -0.43 (`Tuition fees up to date`); todas las demás están entre -0.25 y 0.25. El valor del modelo del baseline (sección 7) va a depender de combinar varias señales moderadas, no de una sola variable dominante.

## 6. Feature engineering

**Atención al leakage:** la partición train/test va antes de ajustar cualquier transformación que dependa de estadísticas del dataset (escalamiento, agrupar categorías raras). `is_portuguese` y `dropout` son excepciones seguras: son recodificaciones fijas de una sola fila (`Nacionality == 1`, `Target == "Dropout"`), no estadísticas agregadas, así que no importa si se calculan antes o después del split.

In [ ]:
from sklearn.model_selection import train_test_split

feature_cols = [
    "Age at enrollment", "Displaced", "Educational special needs", "Debtor",
    "Tuition fees up to date", "Gender", "Scholarship holder", "is_portuguese",
    "Marital status", "Mother's qualification", "Father's qualification",
    "Mother's occupation", "Father's occupation",
]
# Nacionality queda fuera: ya está resumida en is_portuguese (sección 4).
# International queda fuera: es el complemento exacto de is_portuguese (sección 5, r=-1.00).

X = df[feature_cols]
y = df["dropout"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
# stratify=y porque el target está desbalanceado 68/32 (sección 5): sin esto,
# train y test podrían terminar con proporciones de dropout distintas por azar.

In [ ]:
# Transformaciones dentro de un Pipeline: fit solo con train
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numericas = ["Age at enrollment", "Displaced", "Educational special needs", "Debtor",
             "Tuition fees up to date", "Gender", "Scholarship holder", "is_portuguese"]
categoricas = ["Marital status", "Mother's qualification", "Father's qualification",
               "Mother's occupation", "Father's occupation"]

preproceso = ColumnTransformer([
    ("num", StandardScaler(), numericas),
    ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=0.01), categoricas),
])

# Verificación rápida: el preproceso se ajusta solo con X_train
preproceso.fit(X_train)
preproceso.transform(X_train).shape

**Features y su justificación:**

| Feature | Cómo se construyó | Por qué debería ayudar |
|---|---|---|
| `is_portuguese` | `Nacionality == 1` | Colapsa ~20 categorías de nacionalidad (97.5% concentradas en una sola, sección 3) en una señal binaria estable, sin columnas casi vacías |
| Binarias personales (`Displaced`, `Educational special needs`, `Debtor`, `Tuition fees up to date`, `Gender`, `Scholarship holder`) | Tal cual, escaladas con `StandardScaler` | Ya vienen limpias (sección 4); `Tuition fees up to date`, `Debtor` y `Scholarship holder` mostraron la separación más fuerte con el target en la sección 5 |
| `Age at enrollment` | Tal cual, escalada con `StandardScaler` | Segunda correlación más fuerte con dropout (r = 0.25, sección 5) |
| `Marital status`, `Mother's/Father's qualification`, `Mother's/Father's occupation` | `OneHotEncoder(min_frequency=0.01)`: categorías con menos del 1% de las filas de entrenamiento se agrupan en un nivel "infrequent" | Son categóricas nominales de alta cardinalidad con muchos niveles raros (sección 3); agrupar automáticamente evita columnas casi vacías sin necesitar un diccionario semántico completo de los códigos de ocupación (no disponible) |
| Código "Unknown"/"(blank)" en datos de los padres | No se imputa: el `OneHotEncoder` lo trata como un nivel más | Su ausencia está asociada al abandono (73% vs. 32%, sección 4) — convertirlo en su propia categoría preserva esa señal en vez de borrarla |

**Features descartadas:**

| Variable | Por qué se descarta |
|---|---|
| `Nacionality` (cruda) | Reemplazada por `is_portuguese`; no aporta información adicional una vez colapsada |
| `International` | Es el complemento exacto de `is_portuguese` (r = -1.00, sección 5); incluir ambas duplicaría la misma señal |

## 7. Modelo baseline

In [ ]:
# Baseline trivial: DummyClassifier
from sklearn.dummy import DummyClassifier

trivial = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
trivial.fit(X_train, y_train)

In [ ]:
# Modelo propuesto: k-NN, dentro del Pipeline (preproceso se ajusta solo con train)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Barrido ligero de k (no es tuning exhaustivo, solo para no elegir k al azar)
for k in [5, 11, 15, 21, 31, 51]:
    modelo_k = Pipeline([("preproceso", preproceso), ("knn", KNeighborsClassifier(n_neighbors=k))])
    modelo_k.fit(X_train, y_train)
    p_test = modelo_k.predict(X_test)
    print(f"k={k:3d}  test_acc={accuracy_score(y_test, p_test):.3f}  "
          f"test_f1={f1_score(y_test, p_test):.3f}")

# k=11 da el mejor balance precision/recall (F1) sin ser el que más sobreajusta
modelo = Pipeline([
    ("preproceso", preproceso),
    ("knn", KNeighborsClassifier(n_neighbors=11)),
])
modelo.fit(X_train, y_train)

In [ ]:
# Evaluación en train y test. Target desbalanceado (68/32, sección 5): accuracy sola no
# alcanza, se reportan también precision/recall/F1 para la clase dropout (=1)
def reportar(nombre, modelo, X_, y_):
    p = modelo.predict(X_)
    print(f"{nombre:8s} acc={accuracy_score(y_, p):.3f}  "
          f"prec={precision_score(y_, p, zero_division=0):.3f}  "
          f"rec={recall_score(y_, p, zero_division=0):.3f}  "
          f"f1={f1_score(y_, p, zero_division=0):.3f}")

print("--- Trivial ---")
reportar("train", trivial, X_train, y_train)
reportar("test", trivial, X_test, y_test)

print("\n--- k-NN (k=11) ---")
reportar("train", modelo, X_train, y_train)
reportar("test", modelo, X_test, y_test)

**Resultados:**

| Modelo | Accuracy (train) | Accuracy (test) | Precision (test) | Recall (test) | F1 (test) |
|---|---|---|---|---|---|
| Trivial (clase mayoritaria) | 0.679 | 0.679 | 0.000 | 0.000 | 0.000 |
| k-NN (k=11) | 0.786 | 0.768 | 0.711 | 0.468 | 0.565 |

El trivial nunca predice dropout (recall y precision en 0: solo repite "no abandona"), así que su 67.9% de accuracy es un piso, no un logro. El k-NN le gana en accuracy por 8.9 puntos (76.8% vs 67.9%) y, más importante, identifica correctamente el 46.8% de los estudiantes que sí abandonan (recall) con un 71.1% de precisión en esas predicciones — algo que el trivial no puede hacer en absoluto. La brecha train/test (78.6% vs 76.8%) es chica, así que no hay sobreajuste severo.

¿Vale la pena la complejidad extra? Sí, para el propósito del proyecto: una oficina de retención que solo usara el trivial no identificaría a ningún estudiante en riesgo. Pero el recall de 46.8% también dice que el modelo, usando solo información personal y socioeconómica, deja pasar a más de la mitad de quienes sí abandonan — consistente con el hallazgo de la sección 5 de que ninguna variable individual pasa de r=0.43 con el target.

## 8. Conclusiones y límites

*Respondan la pregunta de la sección 1 con lo que encontraron. Después, un párrafo sobre lo que **no** se puede concluir: sesgos de la muestra, variables ausentes, correlaciones que no son causalidad, y en qué contexto este modelo dejaría de funcionar.*

**Respuesta a la pregunta:**

**Límites del análisis:**

**Qué haría falta para responderla mejor:**

## 9. Bitácora de colaboración con AI

*Según la política del curso: si usaron asistencia de AI, documenten en qué partes, qué les pidieron y qué verificaron ustedes. Si no la usaron, escríbanlo también.*

## Anexo: peer critique (se llena en clase)

**Revisor:**

1. ¿Entendí la pregunta del proyecto sin preguntarle al autor?
2. ¿Hay alguna gráfica o celda que yo borraría? ¿Cuál y por qué?
3. ¿Veo algún punto donde se pudo haber colado información del test?